# 01 - Dataset audit (Stage 0)

**Question this notebook answers:** before any feature engineering or modelling, is the
Automatic Speaker Verification Spoofing and Countermeasures Challenge 2019 logical access
(ASVspoof 2019 LA) corpus *safe to model on*? That is, is the bonafide/spoof label
predictable from nuisance metadata alone (duration, sample rate, codec, loudness, silence,
source corpus, speaker identity), rather than from the speech signal itself?

**Mode:** the notebook looks for the ASVspoof 2019 LA download at
`configs/data.yaml: dataset.root`. If it is present (the run stored here prints
`MODE: REAL DATA`), every number below describes the real corpus. If it is absent, the
notebook falls back to a small **synthetic mini-manifest** (synthesised audio plus a fake
protocol file in the exact ASVspoof 2019 LA format) so the same code path still executes; a
verdict produced in that mode is a pipeline check, not a finding about ASVspoof.

Imports only from `audio_df` (`data`, `leakage`); no feature extraction, splitting, or
model code is defined in this notebook.

In [3]:
from __future__ import annotations

import tempfile
import warnings
from pathlib import Path

import pandas as pd
import yaml

from audio_df import data, leakage
from audio_df.synthetic import build_synthetic_asvspoof_tree


def _find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(6):
        if (cur / "pyproject.toml").exists():
            return cur
        cur = cur.parent
    raise RuntimeError(f"could not find repo root (pyproject.toml) from {start}")


REPO_ROOT = _find_repo_root(Path.cwd())
pd.set_option("display.width", 120)

In [4]:
# --- mode detection ---------------------------------------------------------------------
_data_cfg_path = REPO_ROOT / "configs" / "data.yaml"
_real_cfg = yaml.safe_load(_data_cfg_path.read_text())
_real_root = Path(_real_cfg["dataset"]["root"])
if not _real_root.is_absolute():
    _real_root = REPO_ROOT / _real_root

USE_REAL_DATA = _real_root.exists() and any(_real_root.rglob("*.flac"))

if USE_REAL_DATA:
    MODE = "REAL DATA"
    cfg = _real_cfg
    cfg["dataset"]["root"] = str(_real_root)
    cfg["dataset"]["protocol_dir"] = str(REPO_ROOT / cfg["dataset"]["protocol_dir"])
    _tmpdir_handle = None
else:
    MODE = "SYNTHETIC MINI-MANIFEST (no ASVspoof download found)"
    _tmpdir_handle = tempfile.TemporaryDirectory(prefix="asvspoof_synth_")
    cfg = build_synthetic_asvspoof_tree(Path(_tmpdir_handle.name), seed=1337)

print(f"MODE: {MODE}")
print(f"dataset.root = {cfg['dataset']['root']}")

MODE: REAL DATA
dataset.root = /Users/zuttome/LocalDocs/Learning/College/NUS/MTech-AIS/PS/GC2/data/raw/ASVspoof2019/LA


In [5]:
manifest = data.build_manifest(cfg)
print(f"manifest: {len(manifest)} rows, columns = {list(manifest.columns)}")
manifest.head()

manifest: 121461 rows, columns = ['utt_id', 'path', 'label', 'speaker_id', 'attack_id', 'source_corpus', 'sample_rate', 'duration_s', 'codec', 'split', 'language']


,utt_id,path,label,speaker_id,attack_id,source_corpus,sample_rate,duration_s,codec,split,language
0,LA_T_1138215,/Users/zuttome/LocalDocs/Learning/College/NUS/...,bonafide,LA_0079,-,ASVspoof2019_LA,16000,3.458063,FLAC,train,en
1,LA_T_1271820,/Users/zuttome/LocalDocs/Learning/College/NUS/...,bonafide,LA_0079,-,ASVspoof2019_LA,16000,4.395187,FLAC,train,en
2,LA_T_1272637,/Users/zuttome/LocalDocs/Learning/College/NUS/...,bonafide,LA_0079,-,ASVspoof2019_LA,16000,2.899500,FLAC,train,en
3,LA_T_1276960,/Users/zuttome/LocalDocs/Learning/College/NUS/...,bonafide,LA_0079,-,ASVspoof2019_LA,16000,2.812562,FLAC,train,en
4,LA_T_1341447,/Users/zuttome/LocalDocs/Learning/College/NUS/...,bonafide,LA_0079,-,ASVspoof2019_LA,16000,3.533187,FLAC,train,en


## Manifest summary: class / attack / speaker counts

`data.summarise_manifest` groups by (split, label, attack_id) and reports utterance and
distinct-speaker counts per cell.

In [6]:
summary = data.summarise_manifest(manifest)
summary

,split,label,attack_id,n_utterances,n_speakers
0,dev,bonafide,-,2548,20
1,dev,spoof,A01,3716,10
2,dev,spoof,A02,3716,10
3,dev,spoof,A03,3716,10
4,dev,spoof,A04,3716,10
5,dev,spoof,A05,3716,10
6,dev,spoof,A06,3716,10
7,eval,bonafide,-,7355,67
8,eval,spoof,A07,4914,48
9,eval,spoof,A08,4914,48


In [7]:
class_counts = manifest.groupby(["split", "label"])["utt_id"].count().unstack(fill_value=0)
print("Class balance per split:")
class_counts

Class balance per split:


label,bonafide,spoof
split,,
dev,2548,22296
eval,7355,63882
train,2580,22800


### Class balance

The classes are strongly imbalanced in every split (measured above): train has 2,580
bonafide against 22,800 spoof utterances, dev 2,548 against 22,296, and eval 7,355 against
63,882, i.e. roughly one bonafide clip for every nine spoof clips. The imbalance follows from
the protocol design, which pairs one bonafide pool with many attack systems (six in train/dev,
thirteen in eval); it is not a statement about how often spoofs occur in deployment.

Accuracy is therefore uninformative here (always answering "spoof" scores about 90%), and F1
depends on the class ratio. The project reports threshold-free or cost-normalised metrics
instead: the equal error rate (EER), the minimum normalised detection cost function (minDCF),
and the EER per attack system.

### Observation
The summary table shows that train and dev contain only attacks **A01-A06** (3,800 spoof
utterances per attack in train, 3,716 in dev), while eval contains only attacks **A07-A19**
(4,914 utterances per attack), none of which appear in train or dev. Speaker counts are small:
20 bonafide speakers in each of train and dev, 67 bonafide speakers in eval, and 48 speakers
per eval attack.

### Inference
Because train/dev and eval never share an attack system, the official eval split is an
**unseen-attack generalisation test** by construction. A model that scores well on dev (seen
attacks A01-A06) but worse on eval (unseen attacks A07-A19) is not necessarily overfitting in
the usual sense; it may simply be facing spoofing algorithms it has never seen. Any dev/eval
gap must be read with this asymmetry in mind.

## Speaker-overlap check

ASVspoof protocols are speaker-disjoint across train/dev by design. `assert_no_speaker_overlap`
raises (naming the offending speakers) if that is violated -- which would be a serious leak,
since a model could then simply recognise a voice it saw labelled during training.

In [8]:
train_df = manifest[manifest["split"] == "train"].reset_index(drop=True)
dev_df = manifest[manifest["split"] == "dev"].reset_index(drop=True)

try:
    data.assert_no_speaker_overlap(train_df, dev_df)
    print("PASS: no speaker overlap between train and dev.")
except ValueError as exc:
    print(f"FAIL: {exc}")

PASS: no speaker overlap between train and dev.


## Split-content hash

`split_hash` is a stable content hash (independent of row order) so two runs can prove they
used exactly the same split -- computed here twice to show it is deterministic.

In [9]:
h1 = data.split_hash(train_df)
h2 = data.split_hash(train_df.sample(frac=1.0, random_state=0).reset_index(drop=True))
print(f"hash (original order):  {h1}")
print(f"hash (shuffled rows):   {h2}")
assert h1 == h2, "split_hash should be invariant to row order"
print("PASS: split_hash is stable under row reordering.")

hash (original order):  57b1cfa4b521eaa9c320a7ebbd3a7f0ee475ad54ef7875d4ebddcbc67ea37d7d
hash (shuffled rows):   57b1cfa4b521eaa9c320a7ebbd3a7f0ee475ad54ef7875d4ebddcbc67ea37d7d
PASS: split_hash is stable under row reordering.


## Leakage audit (Stage 0)

Fit tiny classifiers (logistic regression, depth-2 decision tree) on **metadata-only**
features (duration, sample rate, codec, source corpus, speaker identity, root-mean-square
(RMS) loudness, and leading/trailing/total silence ratio) with out-of-fold cross-validation,
and report each feature's area under the receiver operating characteristic curve (AUC) and
EER at predicting bonafide/spoof alone. The flag threshold `threshold=0.65` in `leakage.py`
is a **project-local convention, not a literature value** (see the `leakage.py` module
docstring). The one directly comparable published number, Müller et al. 2021
(arXiv:2106.12914), reports a single leading-silence feature reaching 15.1% EER on
ASVspoof 2019 LA eval, well past this threshold, so the convention is conservative without
being arbitrary.

In [10]:
nuisance = leakage.extract_nuisance_features(manifest, audio_root=None)
nuisance.head()

,duration_s,sample_rate,codec,source_corpus,speaker_id,rms,silence_leading,silence_trailing,silence_total
0,3.458063,16000,FLAC,ASVspoof2019_LA,LA_0079,0.096481,0.231343,0.046865,0.333731
1,4.395187,16000,FLAC,ASVspoof2019_LA,LA_0079,0.096049,0.211140,0.053510,0.373861
2,2.899500,16000,FLAC,ASVspoof2019_LA,LA_0079,0.075203,0.220728,0.216417,0.503363
3,2.812562,16000,FLAC,ASVspoof2019_LA,LA_0079,0.104142,0.250306,0.000000,0.261683
4,3.533187,16000,FLAC,ASVspoof2019_LA,LA_0079,0.112211,0.226424,0.076188,0.338841


In [11]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    audit_report = leakage.audit_all(nuisance, manifest["label"], seed=cfg.get("seed", 1337))

for w in caught:
    if issubclass(w.category, UserWarning):
        print(f"WARNING: {w.message}")

audit_report

,feature,auc,auc_ci_lo,auc_ci_hi,eer,eer_ci_lo,eer_ci_hi,model,n,flagged
0,duration_s,0.623945,0.619528,0.628297,0.486262,0.477313,0.495307,tree,121461,False
1,sample_rate,0.499946,0.495129,0.504954,0.600014,0.591958,0.601958,logreg,121461,False
2,codec,0.499946,0.495129,0.504954,0.600014,0.591958,0.601958,logreg,121461,False
3,source_corpus,0.499946,0.495129,0.504954,0.600014,0.591958,0.601958,logreg,121461,False
4,speaker_id,0.649125,0.643018,0.654336,0.414083,0.408754,0.418809,logreg,121461,False
5,rms,0.664084,0.659894,0.668571,0.382682,0.378457,0.387063,logreg,121461,True
6,silence_leading,0.837066,0.834693,0.839672,0.246755,0.243376,0.249563,logreg,121461,True
7,silence_trailing,0.611332,0.606461,0.616454,0.436679,0.432444,0.441117,logreg,121461,False
8,silence_total,0.824814,0.822314,0.827374,0.254992,0.252032,0.258151,logreg,121461,True
9,combined,0.875765,0.873376,0.878086,0.228618,0.225607,0.231621,logreg,121461,True


### Observation
The audit table reports, for every nuisance feature and for the combined-feature model, the
out-of-fold AUC/EER (with bootstrap 95% confidence intervals) at predicting the label from
metadata alone. Measured on all 121,461 utterances:

- **Leading silence** is the strongest single shortcut: AUC 0.837 (EER 0.247).
- **Total silence** is close behind: AUC 0.825 (EER 0.255).
- **RMS loudness** is just over the threshold: AUC 0.664 (EER 0.383).
- **All features combined** reach AUC 0.876 (EER 0.229).
- Speaker identity (AUC 0.649), duration (0.624) and trailing silence (0.611) sit just under
  the threshold and are not flagged.
- Sample rate, codec and source corpus are constant across the corpus (AUC 0.500), so they
  carry no information.

The four flagged entries (`rms`, `silence_leading`, `silence_total`, `combined`) are the ones
named in the `WARNING:` lines above.

### Inference
A detector trained on this corpus could reach a non-trivial score by keying on how much
silence surrounds the speech, or on loudness, instead of on synthetic-speech acoustics. The
leading-silence result is consistent with the shortcut reported by Müller et al. 2021. This
audit does not establish *why* the spoof clips carry less or different silence (for example,
a per-attack generation or post-processing chain); that remains a hypothesis. The practical
consequence is that every downstream benchmark needs a control for silence, which is why the
project evaluates each detector on both raw and silence-trimmed audio (the raw vs trimmed
silence diagnostic).

## Class distributions of the audited nuisance features

Aggregate distributions (never a single cherry-picked example), one panel per feature, class
overlaid.

In [12]:

import matplotlib.pyplot as plt

from audio_df import visualization as viz

plot_cols = [
    c for c in nuisance.columns
    if nuisance[c].notna().any() and nuisance[c].dtype.kind in "fi"
]
n = len(plot_cols)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 3.2))
if n == 1:
    axes = [axes]
for ax, col in zip(axes, plot_cols, strict=False):
    values_by_class = {
        label: nuisance.loc[manifest["label"] == label, col].dropna().to_numpy()
        for label in manifest["label"].unique()
    }
    viz.plot_class_distributions(values_by_class, ax=ax)
    ax.set_title(col)
fig.tight_layout()
plt.show()

/var/folders/hv/lwt64sl51d78tksf3db976yh0000gn/T/ipykernel_11076/3293473292.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### What to look for
Each panel overlays the bonafide and spoof distribution of one numeric nuisance feature. A
panel where the two class curves are clearly separated is a candidate leakage channel; given
the audit table, the leading-silence and total-silence panels are where the largest
separation is expected. The figure is not stored in this notebook's saved output (the
plotting backend was non-interactive when it ran), so re-run this cell to inspect it.

### Inference
A separated distribution here is *consistent with* a flagged AUC above, not independent
evidence; the numeric audit is the primary evidence and this plot is a visual check of the
same claim.

## Verdict

Computed programmatically from the audit table above, then stated in words.

In [13]:
flagged = audit_report[audit_report["flagged"]]
print(f"Mode: {MODE}")
print(f"Flagged features (AUC > 0.65): {list(flagged['feature']) if len(flagged) else 'none'}")
if len(flagged) == 0:
    print("VERDICT: no metadata-only shortcut was found by these tiny classifiers -- "
          "this split is SAFE to proceed to Stage 1/2 modelling from this audit's perspective.")
else:
    print("VERDICT: at least one nuisance feature predicts the label above the project's "
          "convention threshold -- this split should NOT be modelled on as-is until the "
          "flagged feature(s) are investigated (e.g. is it a per-attack recording-chain "
          "artifact, or genuinely inherent to the corpus?).")

Mode: REAL DATA
Flagged features (AUC > 0.65): ['rms', 'silence_leading', 'silence_total', 'combined']
VERDICT: at least one nuisance feature predicts the label above the project's convention threshold -- this split should NOT be modelled on as-is until the flagged feature(s) are investigated (e.g. is it a per-attack recording-chain artifact, or genuinely inherent to the corpus?).


The verdict above is computed on the real corpus (`MODE: REAL DATA`): leading silence, total
silence and RMS loudness each predict the label above the 0.65 convention, so the corpus is
not modelled on as-is. The downstream notebooks address this by reporting raw and
silence-trimmed results side by side. When the notebook runs in synthetic mode instead, the
same verdict cell executes on manufactured data and its output is only a pipeline check.